## Initialization
Run this block once to import all required libraries and initialize classes.

In [ ]:
import uproot
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display
from plotly.subplots import make_subplots
import plotly.graph_objects as go


class Events:
    def __init__(self, root_file_path):
        self.root_file = uproot.open(root_file_path)
        self.event_tree = self.root_file[self.root_file.keys()[1]]          # The raw event tree from the root file
        self._base_var = self.event_tree[0].name                            # The base variable to be plotted. This will be cut by the cutter vars.
        self._cutter_vars = {}                                              # A dict of {variable: range} pairs. Used to store the cutter threshold values for each variable.
        self._original_array = []                                           # The data array associated with the base variable
        self._array_cache = {}                                              # Cache the arrays we use so we don't have to read from the root file each time

    @property
    def cutter_vars(self):
        return self._cutter_vars
    
    @cutter_vars.setter
    def cutter_vars(self, vars=[]):
        '''Takes in a list of cutter variables (strings) and appends them to variables. Pairs them each with a default range that covers 100% of their values.'''
        if isinstance(vars, list):
            for var in vars:
                self.update_dict(var)
        else:
            self.update_dict(vars)
            
    @property
    def base_var(self):
        return self._base_var
    
    @base_var.setter
    def base_var(self, var):
        self._base_var = var
    
    @property
    def original_array(self):
        return self.get_array(self._base_var)

    def get_array(self, var: str):
        if var not in self._array_cache:
            self._array_cache[var] = self.event_tree[var].array(library="np")
        return self._array_cache[var]
    
    def get_max_range(self, var: str):
        '''Returns a tuple of the min and max value in a variable array (original, non-cut data) for determining its full range.'''
        arr = self.get_array(var)
        return (arr.min(), arr.max())

    def set_var_range(self, var: str, range: tuple):
        '''Takes in a variable name string and a min/max (tuple); sets the variable's associated range in the _cutter_vars dict'''
        if var not in self._cutter_vars:
            raise Exception(f"Error: Cutter variable {var} is not in list of variables")
        self._cutter_vars[var] = range
    
    def get_total_mask(self):
        '''Returns a boolean mask based off of all of the range values stored in the _cutter_vars dict'''
        if not self._cutter_vars:
            return np.ones(len(self.get_array(self._base_var)), dtype=bool)

        mask = np.isfinite(self.get_array(next(iter(self._cutter_vars))))
        for var in self._cutter_vars:
            arr = self.get_array(var)
            min_val = self._cutter_vars[var][0]
            max_val = self._cutter_vars[var][1]
            mask &= ((min_val <= arr) & (arr <= max_val))
        
        return mask
    
    def determine_default_vars(self):
        muon_cutters = []
        electron_cutters = []

    def is_bool_variable(self, var: str):
        '''Checks whether or not a cutter variable contains boolean data. If not, we can assume it's all float data instead.'''
        if var not in self._cutter_vars:
            raise Exception(f"Error: Cutter variable {var} is not in list of variables")
        
        for x in self.get_array(var)[:100]:
            if not (x == 1 or x == 0):
                return False
        return True
    
    def update_dict(self, var: str):
        if var not in self._cutter_vars.keys():
            full_range = self.get_max_range(var)
            self._cutter_vars.update({var: full_range})


class UIHelper():
    def __init__(self, events_object):
        self.events = events_object

    def reset(self):
        clear_output()

    def display_function(self):
        events = self.events

        # ── UI controls ──────────────────────────────────────────────────────
        display_variable_dropdown = widgets.Dropdown(
            options=list(self.events.event_tree.keys()),
            description="Display Var:",
        )

        cutter_variable_dropdown = widgets.Dropdown(
            options=list(self.events.event_tree.keys()),
            description="Select cutter:",
        )

        bins_slider = widgets.IntSlider(
            value=300,
            min=100,
            max=1000,
            description="bins",
        )

        # ── Plotly figures (created once, updated in-place) ──────────────────
        base_fig = go.FigureWidget(
            make_subplots(rows=2, cols=1, shared_xaxes=True,
                          subplot_titles=["Original", "After cuts"])
        )
        base_fig.add_trace(go.Bar(name="Original", marker_color="steelblue", marker_line_width=0), row=1, col=1)
        base_fig.add_trace(go.Bar(name="After cuts", marker_color="steelblue", marker_line_width=0), row=2, col=1)
        base_fig.update_layout(showlegend=False, height=500, bargap=0)

        cutter_fig = go.FigureWidget()
        cutter_fig.add_trace(go.Bar(name="Data", marker_color="steelblue", marker_line_width=0))
        cutter_fig.update_layout(
            title="Data Selection",
            showlegend=False,
            height=400,
            bargap=0,
        )

        # ── Plot update functions ─────────────────────────────────────────────
        def draw_base_plot():
            var = display_variable_dropdown.value
            bins = bins_slider.value
            original_arr = events.get_array(var)
            mask = events.get_total_mask()
            cut_arr = original_arr[mask]
            x_range = events.get_max_range(var)

            orig_counts, bin_edges = np.histogram(original_arr, bins=bins, range=x_range)
            cut_counts, _ = np.histogram(cut_arr, bins=bins, range=x_range)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

            with base_fig.batch_update():
                base_fig.data[0].x = bin_centers
                base_fig.data[0].y = orig_counts
                base_fig.data[1].x = bin_centers
                base_fig.data[1].y = cut_counts
                base_fig.layout.xaxis.title.text = var
                base_fig.layout.xaxis2.title.text = var

        def draw_cutter_plot(var: str):
            range_vals = events.cutter_vars[var]   # current slider selection
            full_range = events.get_max_range(var) # full extent of the data

            counts, bin_edges = np.histogram(events.get_array(var), bins=500, range=full_range)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

            with cutter_fig.batch_update():
                cutter_fig.data[0].x = bin_centers
                cutter_fig.data[0].y = counts
                cutter_fig.layout.title.text = f"Data Selection: {var}"
                cutter_fig.layout.xaxis.title.text = var
                cutter_fig.layout.update(shapes=[
                    # Grey out left exclusion region
                    dict(type="rect", x0=full_range[0], x1=range_vals[0],
                         y0=0, y1=1, xref="x", yref="paper",
                         fillcolor="grey", opacity=0.3, line_width=0, layer="below"),
                    # Grey out right exclusion region
                    dict(type="rect", x0=range_vals[1], x1=full_range[1],
                         y0=0, y1=1, xref="x", yref="paper",
                         fillcolor="grey", opacity=0.3, line_width=0, layer="below"),
                    # Cut lines
                    dict(type="line", x0=range_vals[0], x1=range_vals[0],
                         y0=0, y1=1, xref="x", yref="paper",
                         line=dict(color="red", dash="dash")),
                    dict(type="line", x0=range_vals[1], x1=range_vals[1],
                         y0=0, y1=1, xref="x", yref="paper",
                         line=dict(color="red", dash="dash")),
                ])

        # ── Slider observer ───────────────────────────────────────────────────
        def attach_slider_observer(slider, var):
            def on_slider_change(change):
                events.set_var_range(var, change["new"])
                draw_cutter_plot(var)
                draw_base_plot()        # <-- add this
            slider.observe(on_slider_change, names="value")

        # ── Cutter tab ────────────────────────────────────────────────────────
        def draw_cutter_widget():
            vars = events.cutter_vars
            children = []
            for var in vars:
                child_range = events.get_max_range(var)
                slider = widgets.FloatRangeSlider(
                    value=child_range,
                    min=child_range[0],
                    max=child_range[1],
                    description=var,
                )
                children.append(slider)
                attach_slider_observer(slider, var)

            cutter_tab = widgets.Tab()
            cutter_tab.children = children
            cutter_tab.titles = list(vars.keys())

            def on_tab_change(change):
                draw_cutter_plot(cutter_tab.get_title(change["new"]))

            cutter_tab.observe(on_tab_change, names="selected_index")

            # Pre-cache arrays and draw initial cutter plot
            for var in events.cutter_vars:
                events.get_array(var)

            if children:
                draw_cutter_plot(list(vars.keys())[0])

            return (cutter_tab, children)

        def update_cutter_widget(var: str, cutter_tuple):
            tab_widget = cutter_tuple[0]
            children_list = cutter_tuple[1]

            existing_titles = [tab_widget.get_title(i) for i in range(len(tab_widget.children))]
            if var not in existing_titles:
                events.cutter_vars = var
                child_range = events.get_max_range(var)
                new_child = widgets.FloatRangeSlider(
                    value=child_range,
                    min=child_range[0],
                    max=child_range[1],
                    description=var,
                )
                children_list.append(new_child)
                attach_slider_observer(new_child, var)
                tab_widget.children = children_list
                tab_widget.set_title(len(tab_widget.children) - 1, var)

            draw_cutter_plot(var)

        # ── Wire up observers ─────────────────────────────────────────────────
        cutter_tuple = draw_cutter_widget()

        display_variable_dropdown.observe(lambda _: draw_base_plot(), names="value")
        bins_slider.observe(lambda _: draw_base_plot(), names="value")
        cutter_variable_dropdown.observe(
            lambda change: update_cutter_widget(change["new"], cutter_tuple),
            names="value",
        )

        # Initial base plot draw
        draw_base_plot()

        # ── Layout ────────────────────────────────────────────────────────────
        left_panel = widgets.VBox([display_variable_dropdown, bins_slider, base_fig])
        right_panel = widgets.VBox([cutter_variable_dropdown, cutter_tuple[0], cutter_fig])

        display(widgets.HBox([left_panel, right_panel]))


In [45]:
## temporary test block, to be deleted once the classes are functional

events = Events("data/00385270_00000001_1.dvntuple.root")

events.cutter_vars = ["B0_P", "B0_ENDVERTEX_CHI2"]

events.set_var_range("B0_P", (60000,70000))
events.set_var_range("B0_ENDVERTEX_CHI2", (0,70))

mask = events.get_total_mask()

ui_helper = UIHelper(events)

#ui_helper.reset()
ui_helper.display_function()


## File selection
Run this block to select the root file you'd like to use for cutting.
You can change your file selection without needing to re-run this block.

In [3]:
fc = FileChooser("data/")
fc.filter_pattern = "*.root"

display(fc)

FileChooser(path='/Users/elliotlyons/Desktop/Classwork/CERN Project PRO1000/Project-116/data', filename='', ti…

## Cutting

In [4]:
print(f"Opening file: {fc.selected_filename}")
file = uproot.open(fc.selected)

print(f"All file keys: {file.keys()}")

keys = file.keys()[1]   
tree = file[keys]
print(f"Tree-specific keys: {tree.keys()}")

# TODO: if/else statement for checking for the muon or electron case to provide an appropriate default cutter variable list 

#reading branches from the list
all_branches = tree.keys()
variables = list(all_branches)

###Conditions###
#Add new functions and limits here then change only the combine_masks function to add new cuts to the final plot
#rest should be unchanged

def kaon_probability_mask(k_threshold=0.85): # change the treshold if you want more or less stict cuts
    kplus = tree["Kplus_ProbNNk"].array(library="np")
    #kminus = tree["Kmin_ProbNNk"].array(library="np")
    #kstar = tree["kst_ProbNNk"].array(library="np")

    kplus_mask = np.isfinite(kplus) & (kplus > k_threshold)
    #kminus_mask = np.isfinite(kminus) & (kminus > k_threshold)
    #kstar_mask = np.isfinite(kstar) & (kstar > k_threshold)
    return kplus_mask 

def pminus_probability_mask(pmin_threshold=0.1): # change the treshold if you want more or less stict cuts
    pminus = tree["piminus_ProbNNpi"].array(library="np")
    pminus_mask = np.isfinite(pminus) & (pminus > pmin_threshold)
    return pminus_mask

#checking if it is not a muon
def mu_isMuon_mask():
    muplus_isMuon = tree["muplus_isMuon"].array(library="np")
    muplus_mask = np.isfinite(muplus_isMuon) & (muplus_isMuon != 0)

    muminus_isMuon = tree["muminus_isMuon"].array(library="np")
    muminus_mask = np.isfinite(muminus_isMuon) & (muminus_isMuon != 0)
    return muplus_mask & muminus_mask

def b0_flight_distance_mask(min_fdchi2=100):

    flight_chi2 = tree["B0_FDCHI2_OWNPV"].array(library="np")
    flight_distance_chi2_mask = np.isfinite(flight_chi2) & (flight_chi2 > min_fdchi2)
    return (flight_distance_chi2_mask)

#combine conditions -add new masks to variable list here
def combine_masks(k_threshold=0.85, pmin_threshold=0.1, min_fdchi2=100):
    return (kaon_probability_mask(k_threshold) 
            & pminus_probability_mask(pmin_threshold) 
            & mu_isMuon_mask() 
            & b0_flight_distance_mask(min_fdchi2))
       


###Plotting two graphs next to each other
def plot_with_cuts(variable, bins, x_min, x_max):

    # Load raw data (never modify this)
    original_data = tree[variable].array(library="np")

    xlabel = variable

    # compute data for the right graph 
    mask = combine_masks()
    filtered_data = original_data[mask]

    # Two graphs side by side
    fig, axes = plt.subplots(2, 1, figsize=(5, 5), sharey=True)

    axes[0].hist(original_data, bins=bins, range=(x_min, x_max), histtype="step")
    axes[0].set_title("Original")
    axes[0].set_xlabel(xlabel)
    axes[0].set_ylabel("Number of events")

    axes[1].hist(filtered_data, bins=bins, range=(x_min, x_max), histtype="step",color="orange")
    axes[1].set_title(f"After cuts: ")
    axes[1].set_xlabel(xlabel)
    axes[1].set_ylabel("Number of events")

    #plt.yscale("log")
    plt.tight_layout()
    plt.show()

    print("Original events:", len(original_data))
    print("Events after cut:", len(filtered_data))
    print("Removed events:", len(original_data) - len(filtered_data))


###Interactive part###
#choose variable to plot
display_variable_dropdown = widgets.Dropdown(
    options=variables,
    description="Display Variable:"
)

cutter_variable_dropdown = widgets.Dropdown(
    options = variables,
    description = "Select cutter variables:"
)

#sliders for cuts
bins_slider = widgets.IntSlider(
    value=100,
    min=1,
    max=300,
    step=1,
    description="Bins:"
)
k_threshold_slider = widgets.FloatSlider(
    value=0.85,
    min=0,
    max=1,
    step=0.01,
    description="K+ threshold:"
)
pmin_threshold_slider = widgets.FloatSlider(
    value=0.1,
    min=0,
    max=1,
    step=0.01,
    description="pi- threshold:"
)
fdchi2_threshold_slider = widgets.FloatSlider(
    value=100,
    min=0,
    max=500,
    step=1,
    description="Flight Distance CHI2 threshold:"
)

cutter_variables = ["B0_P", "B0_ENDVERTEX_CHI2"]

def add_cutter_var(var):
    if var not in cutter_variables:
        cutter_variables.append(var)
    update_cutter_widget()



def update_cutter_widget():
    children = []
    for var in cutter_variables:
        var_array = tree[var].array(library="np")
        child_range = (var_array.min(), var_array.max())

        widget_child = widgets.FloatRangeSlider(value=child_range,
                                               min=child_range[0],
                                               max=child_range[1],
                                               description=var
                                            )
        children.append(widget_child)

    cutter_tab = widgets.Tab()
    cutter_tab.children = children
    cutter_tab.titles = cutter_variables

    display(cutter_tab)


# Runtime interactivity

out1 = widgets.Output()
out2 = widgets.Output()


with out2:
    widgets.interact(
        add_cutter_var,
        var=cutter_variable_dropdown
    )

with out1:
    widgets.interact(
        plot_with_cuts,
        variable=display_variable_dropdown,
        bins=bins_slider,
        x_min=widgets.FloatText(value=0, description="x min"),
        x_max=widgets.FloatText(value=10000, description="x max")
    )


widgets.TwoByTwoLayout(top_left=out1, top_right=out2)


Opening file: 00385270_00000001_1.dvntuple.root
All file keys: ['MyDecayTree_muons;1', 'MyDecayTree_muons/DecayTree;1', 'GetIntegratedLuminosity;1', 'GetIntegratedLuminosity/LumiTuple;1']
Tree-specific keys: ['B0_MCORR', 'B0_MCORRERR', 'B0_MCORRVERTEXERR', 'B0_PTREL', 'B0_ENDVERTEX_X', 'B0_ENDVERTEX_Y', 'B0_ENDVERTEX_Z', 'B0_ENDVERTEX_XERR', 'B0_ENDVERTEX_YERR', 'B0_ENDVERTEX_ZERR', 'B0_ENDVERTEX_CHI2', 'B0_ENDVERTEX_NDOF', 'B0_ENDVERTEX_COV_', 'B0_OWNPV_X', 'B0_OWNPV_Y', 'B0_OWNPV_Z', 'B0_OWNPV_XERR', 'B0_OWNPV_YERR', 'B0_OWNPV_ZERR', 'B0_OWNPV_CHI2', 'B0_OWNPV_NDOF', 'B0_OWNPV_COV_', 'B0_IP_OWNPV', 'B0_IPCHI2_OWNPV', 'B0_FD_OWNPV', 'B0_FDCHI2_OWNPV', 'B0_DIRA_OWNPV', 'B0_P', 'B0_PT', 'B0_PE', 'B0_PX', 'B0_PY', 'B0_PZ', 'B0_MM', 'B0_MMERR', 'B0_M', 'B0_ID', 'B0_TAU', 'B0_TAUERR', 'B0_TAUCHI2', 'B0_L0Global_Dec', 'B0_L0Global_TIS', 'B0_L0Global_TOS', 'B0_Hlt1Global_Dec', 'B0_Hlt1Global_TIS', 'B0_Hlt1Global_TOS', 'B0_Hlt1Phys_Dec', 'B0_Hlt1Phys_TIS', 'B0_Hlt1Phys_TOS', 'B0_Hlt2Global_De

TwoByTwoLayout(children=(Output(layout=Layout(grid_area='top-left')), Output(layout=Layout(grid_area='top-righ…